# Day10 — 로컬 이미지 → 브랜드 스타일 한국어 광고 카피 자동 생성

> **목적**: `./images/` 폴더에 직접 저장한 무신사 제품 이미지를 로드하고,  
> **Claude Vision** 으로 메타 자동 추출 후  
> **Claude Sonnet** 으로 Style A (MUJI형) / Style B (UNIQLO형) 광고 카피를 자동 생성.  
> Day11 SFT 학습 데이터 (450쌍) 의 핵심 파이프라인.

---

## 흐름 (S0 ~ S10)

```
S0  환경 설정 + Claude API 초기화
S1  base64 이미지 인코딩 유틸
S2  로컬 이미지 폴더 스캔 + Claude Vision 메타 자동 추출
S3  스타일(페르소나) 정의 — Style A (MUJI형) / Style B (UNIQLO형)
S4  카피 품질 라벨 빌더 (instruction 포맷용)
S5  스타일 × 이미지 Vision 카피 프롬프트 + 1건 시연
S6  비동기 배치 생성 + 토큰/비용 측정
S7  Claude-Judge 채점 (style_fit / copy_quality / brand_match)
S8  DPO chosen/rejected 자동 생성 (Rejection Sampling)
S9  dedup + jsonl 저장 + SFT 포맷 호환 검증
S10 비용 외삽
```

## 이미지 파일 명명 규칙

```
images/
├── tops_001.jpg        # 상의
├── tops_002.jpg
├── bottoms_001.jpg     # 하의
├── outer_001.jpg       # 아우터
├── shoes_001.jpg       # 신발
└── acc_001.jpg         # 액세서리
```

파일명 형식: `{category}_{번호}.jpg`  
카테고리 prefix → `tops`, `bottoms`, `outer`, `shoes`, `acc`, `bag` 등

## 데이터 포맷 (Day11 SFT 입력)

```json
{
  "image": "images/tops_001.jpg",
  "instruction": "이 제품 사진을 보고 MUJI 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
  "output": "가볍고 부드러운 소재로 하루 종일 편안하게 입을 수 있습니다."
}
```

## S0 — 환경 설정 + Claude API 초기화

In [ ]:
# 첫 실행 시만 활성화
!pip install -q langchain==0.3.13 langchain-core==0.3.28
!pip install -q langchain-anthropic
!pip install -q pydantic==2.10.4
!pip install -q pillow nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.28 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.28 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/코리아아케데미_ai_포트폴리오/muji /content/muji
!cp -r /content/drive/MyDrive/코리아아케데미_ai_포트폴리오/uniqlo /content/uniqlo

In [ ]:
import os, random, time, json, asyncio
import numpy as np
from pathlib import Path
from getpass import getpass
import nest_asyncio
nest_asyncio.apply()

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ── Claude API 초기화 ──────────────────────────────────────
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

from langchain_anthropic import ChatAnthropic

llm     = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.5)
llm_hot = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=1.0)  # DPO 다양성용
MODEL_NAME = "claude-sonnet-4-20250514"
PRICE_IN, PRICE_OUT = 3.0, 15.0   # $/1M tokens

print(f"[OK] model={MODEL_NAME}  (in ${PRICE_IN}/1M, out ${PRICE_OUT}/1M)")

ANTHROPIC_API_KEY: ··········
[OK] model=claude-sonnet-4-20250514  (in $3.0/1M, out $15.0/1M)


In [ ]:
# ── 토큰·비용 카운터 (Claude 전용) ───────────────────────
import contextlib as _ctxlib
from langchain_core.callbacks import BaseCallbackHandler

class _TokenCounter:
    def __init__(self, pin, pout):
        self.prompt_tokens = 0; self.completion_tokens = 0; self.total_cost = 0.0
        self._pin = pin / 1_000_000; self._pout = pout / 1_000_000
    def _add(self, i, o):
        i, o = int(i or 0), int(o or 0)
        self.prompt_tokens += i; self.completion_tokens += o
        self.total_cost += i * self._pin + o * self._pout

class _AnthropicCB(BaseCallbackHandler):
    def __init__(self, c): self.c = c
    def on_llm_end(self, resp, **kw):
        for gens in (resp.generations or []):
            for g in gens:
                meta = getattr(getattr(g, "message", None), "usage_metadata", None)
                if meta:
                    self.c._add(meta.get("input_tokens", 0), meta.get("output_tokens", 0))
                    return

@_ctxlib.contextmanager
def make_cb_ctx():
    counter = _TokenCounter(PRICE_IN, PRICE_OUT)
    handler = _AnthropicCB(counter)
    p1 = list(getattr(llm, "callbacks", None) or [])
    p2 = list(getattr(llm_hot, "callbacks", None) or [])
    llm.callbacks = p1 + [handler]; llm_hot.callbacks = p2 + [handler]
    try: yield counter
    finally: llm.callbacks = p1; llm_hot.callbacks = p2

print("[OK] 토큰 카운터 준비")

[OK] 토큰 카운터 준비


## S1 — base64 이미지 인코딩 유틸

In [ ]:
import base64
from io import BytesIO
from PIL import Image


def pil_to_data_url(img: Image.Image, fmt: str = "JPEG") -> str:
    """PIL.Image → data:image/jpeg;base64,... URL (LangChain HumanMessage 용)."""
    buf = BytesIO()
    img.convert("RGB").save(buf, format=fmt, quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/{fmt.lower()};base64,{b64}"


# 동작 확인
dummy = Image.new("RGB", (128, 128), color=(200, 200, 200))
url = pil_to_data_url(dummy)
print(f"[OK] data URL 길이: {len(url):,} chars")

[OK] data URL 길이: 1,203 chars


## S2 — 로컬 이미지 폴더 스캔 + Claude Vision 메타 자동 추출

### 폴더 구조 (방법 2: 브랜드별 서브폴더)
```
images/
├── muji/
│   ├── tops_001.jpg
│   ├── outer_001.jpg
│   └── ...
└── uniqlo/
    ├── tops_001.jpg
    ├── bottoms_001.jpg
    └── ...
```

### product_id 형식
```
{brand}_{category}_{번호}   예) muji_tops_001, uniqlo_outer_002
```

### SFT image 경로 포맷
```json
{ "image": "images/muji/tops_001.jpg", ... }
{ "image": "images/uniqlo/tops_001.jpg", ... }
```

### Claude Vision 메타 추출 항목
| 항목 | 내용 |
|------|------|
| `product_name` | 제품 유형 + 특징 요약 |
| `brand_guess` | 이미지 스타일로 추정한 브랜드 계열 |
| `color` | 주요 색상 |
| `material_guess` | 소재 추정 |
| `tags` | 스타일 키워드 3~5개 |
| `description` | 카피 생성에 사용할 제품 설명 1~2문장 |


In [ ]:
import re
from pathlib import Path
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ── 브랜드별 서브폴더 방식 (방법 2) ─────────────────────────────
# Google Drive에서 복사한 폴더 구조:
#   images/
#     muji/    ← /content/muji 복사본
#     uniqlo/  ← /content/uniqlo 복사본
IMAGES_ROOT = Path("./images")
BRAND_DIRS = {
    "muji":   IMAGES_ROOT / "muji",
    "uniqlo": IMAGES_ROOT / "uniqlo",
}

# 폴더 생성 (없으면 만들기)
IMAGES_ROOT.mkdir(exist_ok=True)
for brand, d in BRAND_DIRS.items():
    d.mkdir(parents=True, exist_ok=True)

# Drive에서 복사한 폴더를 images/ 하위로 이동 (최초 1회만)
import shutil
for brand, src in [("muji", Path("/content/muji")), ("uniqlo", Path("/content/uniqlo"))]:
    dst = IMAGES_ROOT / brand
    if src.exists() and not any(dst.iterdir()) if dst.exists() else src.exists():
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        print(f"[OK] /content/{brand} → images/{brand} 복사 완료")
    else:
        print(f"[INFO] images/{brand} 이미 존재하거나 /content/{brand} 없음 — 스킵")

# ── 카테고리 매핑 (파일명 prefix → 한글 카테고리) ──────────────
CATEGORY_MAP = {
    "top":    "상의",
    "bottom": "하의",
    "outer":  "아우터",
    "acc":    "액세서리",
    "bag":    "가방",
}

def parse_category_from_filename(filename: str) -> tuple[str, str]:
    """
    'tops_001.jpg' → (category_en='tops', category_ko='상의')
    알 수 없는 prefix → ('etc', '기타')
    """
    stem = Path(filename).stem
    prefix = stem.split("_")[0].lower()
    category_ko = CATEGORY_MAP.get(prefix, "기타")
    return prefix, category_ko


# ── Claude Vision 메타 추출 스키마 ──────────────────────────
class ProductMeta(BaseModel):
    product_name:   str       = Field(description="제품 유형 + 특징 (예: 오버핏 린넨 셔츠)")
    brand_guess:    str       = Field(description="이미지 스타일로 추정한 브랜드 계열 (확실치 않으면 '스트릿', '미니멀' 등 스타일 계열로)")
    color:          str       = Field(description="주요 색상 (예: 오프화이트, 차콜그레이)")
    material_guess: str       = Field(description="소재 추정 (예: 면, 린넨, 데님)")
    tags:           list[str] = Field(description="스타일 키워드 3~5개 (예: ['오버핏', '캐주얼', '베이직'])")
    description:    str       = Field(description="카피 생성에 활용할 제품 설명 1~2문장")

meta_parser = JsonOutputParser(pydantic_object=ProductMeta)

meta_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한국 패션 상품 분석 전문가입니다.\n"
     "제품 이미지를 보고 아래 JSON 형식으로 메타 정보를 추출하세요.\n"
     "브랜드가 확실하지 않으면 스타일 계열로 표현하세요 (예: 미니멀, 스트릿, 캐주얼).\n\n"
     "{format_instructions}"
    ),
    ("user", [
        {"type": "text",      "text": "카테고리 힌트: {category_ko}\n위 이미지의 제품 메타 정보를 추출해주세요."},
        {"type": "image_url", "image_url": {"url": "{image_data_url}"}},
    ]),
]).partial(format_instructions=meta_parser.get_format_instructions())

meta_chain = meta_prompt | llm | meta_parser

print("[OK] 메타 추출 체인 정의 완료")
for brand, d in BRAND_DIRS.items():
    imgs = list(d.glob("*.jpg")) + list(d.glob("*.jpeg")) + list(d.glob("*.png")) + list(d.glob("*.webp"))
    print(f"[INFO] images/{brand}/ → {len(imgs)}개 이미지")


[OK] /content/muji → images/muji 복사 완료
[OK] /content/uniqlo → images/uniqlo 복사 완료
[OK] 메타 추출 체인 정의 완료
[INFO] images/muji/ → 37개 이미지
[INFO] images/uniqlo/ → 37개 이미지


In [ ]:
# ── 브랜드별 재귀 이미지 스캔 ───────────────────────────────
EXTS = {".jpg", ".jpeg", ".png", ".webp"}

image_files = []   # (brand, Path) 튜플 리스트
for brand, d in BRAND_DIRS.items():
    brand_imgs = sorted(p for p in d.iterdir() if p.suffix.lower() in EXTS)
    for p in brand_imgs:
        image_files.append((brand, p))

print(f"[INFO] 전체 이미지: {len(image_files)}개")
print("-" * 60)

brand_counts = {}
for brand, f in image_files:
    prefix, cat_ko = parse_category_from_filename(f.name)
    brand_counts.setdefault(brand, 0)
    brand_counts[brand] += 1
    print(f"  [{brand:<6}] {f.name:<30} → category={cat_ko}")

print()
for brand, cnt in brand_counts.items():
    print(f"  {brand}: {cnt}개")

if len(image_files) == 0:
    print("\n⚠️  images/muji/ 와 images/uniqlo/ 가 비어 있습니다.")
    print("   Google Drive에서 복사가 완료됐는지 확인하세요.")


[INFO] 전체 이미지: 74개
------------------------------------------------------------
  [muji  ] acc_001.webp                   → category=액세서리
  [muji  ] acc_002.webp                   → category=액세서리
  [muji  ] bag_001.webp                   → category=가방
  [muji  ] bag_002.webp                   → category=가방
  [muji  ] bag_003.webp                   → category=가방
  [muji  ] bag_004.webp                   → category=가방
  [muji  ] bag_005.webp                   → category=가방
  [muji  ] bottom_001.webp                → category=하의
  [muji  ] bottom_002.webp                → category=하의
  [muji  ] bottom_003.webp                → category=하의
  [muji  ] bottom_004.webp                → category=하의
  [muji  ] bottom_005.webp                → category=하의
  [muji  ] bottom_006.webp                → category=하의
  [muji  ] bottom_007.webp                → category=하의
  [muji  ] bottom_008.webp                → category=하의
  [muji  ] bottom_009.webp                → category=하의
  [muji  ] bottom_01

In [ ]:
# ── PIL 로딩 + Claude Vision 메타 추출 ──────────────────────
# META_EXTRACT_LIMIT: 비용 절감을 위해 처음 N개만 추출. 전체는 len(image_files)
META_EXTRACT_LIMIT = min(74, len(image_files))

PRODUCT_SEEDS = []
data_urls = []
failed = []

print(f"[INFO] 메타 추출 대상: {META_EXTRACT_LIMIT}개")
print("-" * 60)

for idx, (brand, img_path) in enumerate(image_files[:META_EXTRACT_LIMIT]):
    # 1. PIL 로딩
    try:
        pil_img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"  [{idx+1}] ❌ PIL 로딩 실패: {img_path.name} — {e}")
        failed.append(img_path.name)
        continue

    # 2. 파일명에서 카테고리 파싱
    category_en, category_ko = parse_category_from_filename(img_path.name)
    # product_id: 브랜드_파일명stem (예: muji_tops_001)
    product_id = f"{brand}_{img_path.stem}"

    # 3. Claude Vision 메타 추출
    data_url = pil_to_data_url(pil_img)
    try:
        meta = meta_chain.invoke({
            "category_ko":    category_ko,
            "image_data_url": data_url,
        })
    except Exception as e:
        print(f"  [{idx+1}] ❌ 메타 추출 실패: {img_path.name} — {e}")
        failed.append(img_path.name)
        continue

    # 4. PRODUCT_SEEDS 에 추가
    seed = {
        "product_id":   product_id,               # muji_tops_001
        "brand":        brand,                     # "muji" or "uniqlo"
        "product_name": meta.get("product_name", ""),
        "category":     category_ko,
        "category_en":  category_en,
        "color":        meta.get("color", ""),
        "material":     meta.get("material_guess", ""),
        "tags":         meta.get("tags", []),
        "description":  meta.get("description", ""),
        "image_path":   str(img_path),             # 절대경로
        "image_rel":    f"images/{brand}/{img_path.name}",  # SFT 저장용 상대경로
        "pil_image":    pil_img,
    }
    PRODUCT_SEEDS.append(seed)
    data_urls.append(data_url)

    print(f"  [{idx+1}/{META_EXTRACT_LIMIT}] ✅ [{brand}] {img_path.name:<25}"
          f"| {meta.get('product_name','')[:20]:<20}"
          f"| {meta.get('color',''):<12}"
          f"| tags={meta.get('tags',[])}")
    time.sleep(0.3)

print(f"\n{'='*60}")
print(f"[OK] PRODUCT_SEEDS: {len(PRODUCT_SEEDS)}건 완료")
if failed:
    print(f"[WARN] 실패: {failed}")

# ── 브랜드 × 카테고리 분포 확인 ─────────────────────────────
from collections import Counter
print("\n브랜드 × 카테고리 분포:")
dist = Counter((s['brand'], s['category']) for s in PRODUCT_SEEDS)
for (brand, cat), cnt in sorted(dist.items()):
    print(f"  [{brand:<6}] {cat:<6}: {'█' * cnt} ({cnt}건)")

# 샘플
if PRODUCT_SEEDS:
    s0 = PRODUCT_SEEDS[0]
    print(f"\n[샘플 0: {s0['product_id']}]")
    print(f"  브랜드:   {s0['brand']}")
    print(f"  제품명:   {s0['product_name']}")
    print(f"  카테고리: {s0['category']}")
    print(f"  이미지경로: {s0['image_rel']}")


[INFO] 메타 추출 대상: 74개
------------------------------------------------------------
  [1/74] ✅ [muji] acc_001.webp             | 클래식 라운드 안경          | 블랙          | tags=['클래식', '라운드', '미니멀', '베이직', '빈티지']
  [2/74] ✅ [muji] acc_002.webp             | 엘라스틱 브레이드 벨트        | 블랙          | tags=['엘라스틱', '브레이드', '캐주얼', '베이직', '실버버클']
  [3/74] ✅ [muji] bag_001.webp             | 대용량 폴딩 더플백          | 블랙          | tags=['대용량', '폴딩', '여행용', '실용적', '미니멀']
  [4/74] ✅ [muji] bag_002.webp             | 심플 에코백              | 내추럴 베이지     | tags=['에코백', '심플', '베이직', '캐주얼', '데일리']
  [5/74] ✅ [muji] bag_003.webp             | 미니 백팩 키링 파우치        | 차콜그레이       | tags=['미니백팩', '키링', '파우치', '실용적', '캐주얼']
  [6/74] ✅ [muji] bag_004.webp             | 미니멀 호보백             | 블랙          | tags=['호보백', '미니멀', '캐주얼', '데일리', '심플']
  [7/74] ✅ [muji] bag_005.webp             | 미니멀 크로스백            | 라이트그레이      | tags=['미니멀', '크로스백', '심플', '데일리', '베이직']
  [8/74] ✅ [muji] bottom_001.webp          | 루즈핏 하프 팬츠          

## S3 — 스타일(페르소나) 정의

| 스타일 | 톤 | 예시 카피 |
|--------|-----|-----------|
| **Style A (MUJI형)** | 간결·여백·기능 중심 | "가볍고 부드러운 소재로 하루 종일 편안하게 입을 수 있습니다" |
| **Style B (UNIQLO형)** | 감각적·일상·라이프스타일·기능미 | "매일 입고 싶은 옷이 있다면, 바로 이겁니다." |

In [ ]:
from dataclasses import dataclass, field


@dataclass
class StyleSpec:
    name:        str
    style_id:    str           # "A" or "B"
    tone_hint:   str           # Claude 호출 시 톤 가이드
    examples:    list          # 예시 카피 (프롬프트 few-shot 용)
    instruction_template: str  # SFT instruction 포맷
    negative_patterns: list = field(default_factory=list)  # DPO rejected 생성용


STYLE_A = StyleSpec(
    name    = "MUJI형",
    style_id = "A",
    tone_hint = (
        "MUJI (무인양품) 스타일의 한국어 광고 카피. "
        "간결하고 여백 있는 문장. 기능/소재/편안함 중심. "
        "수식어 최소화. 마침표로 끝나는 한 문장. 50자 이내. "
        "감탄사·슬래시·이모지 절대 금지. 조용하고 신뢰감 있는 톤."
    ),
    examples = [
        "가볍고 부드러운 소재로 하루 종일 편안하게 입을 수 있습니다.",
        "오래 입을수록 몸에 맞춰지는 면 소재입니다.",
        "군더더기 없는 실루엣으로 어디서든 자연스럽습니다.",
        "계절이 바뀌어도 변하지 않는 기본에 충실한 디자인입니다.",
    ],
    instruction_template = "이 제품 사진을 보고 MUJI 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
    negative_patterns = [
        "과장된 표현 (최고, 완벽, 혁신적 등) 을 넣어 홈쇼핑 스타일로 작성하세요.",
        "'~해요', '~죠', '~잖아요' 같은 구어체 + 이모지를 섞어 작성하세요.",
        "3문장 이상 길게, SNS 해시태그를 달아 작성하세요.",
    ],
)

STYLE_B = StyleSpec(
    name    = "UNIQLO형",
    style_id = "B",
    tone_hint = (
        "유니클로(UNIQLO) LifeWear 브랜드 스타일의 한국어 광고 카피. "
        "일상 속 편안함과 라이프스타일을 부드럽게 제안. "
        "소재·기능을 감성적으로 표현 ('매일 입고 싶은', '어디서든 어울리는'). "
        "따뜻하고 친근한 구어체 (~합니다, ~이겁니다). "
        "과장·직설·스트릿 슬랭 없이 세련되게. 35자 이내."
    ),
    examples = [
        "매일 입고 싶은 옷이 있다면, 바로 이겁니다.",
        "감각적인 디자인, 끝없는 편안함.",
        "어디서든 어울리는 옷 한 벌이 하루를 가볍게 만듭니다.",
        "작은 디테일이 하루를 조금 더 특별하게 해줍니다.",
        "입는 순간 알게 되는 편안함이 있습니다.",
    ],
    instruction_template = "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
    negative_patterns = [
        "MUJI처럼 조용하고 기능만 서술하는 문어체로 작성하세요.",
        "스트릿 반말 (진심, 레전드, 헐) 을 섞어 짧게 작성하세요.",
        "공식적이고 딱딱한 존댓말로 50자 이상 길게 작성하세요.",
    ],
)

STYLES = {"A": STYLE_A, "B": STYLE_B}

for sid, spec in STYLES.items():
    print(f"  Style {sid} ({spec.name}): {spec.tone_hint[:60]}...")

  Style A (MUJI형): MUJI (무인양품) 스타일의 한국어 광고 카피. 간결하고 여백 있는 문장. 기능/소재/편안함 중심. 수식어...
  Style B (UNIQLO형): 유니클로(UNIQLO) LifeWear 브랜드 스타일의 한국어 광고 카피. 일상 속 편안함과 라이프스타일을 ...


## S4 — 카피 품질 라벨 빌더

In [ ]:
def build_product_context(seed: dict) -> str:
    """
    제품 메타 → 프롬프트에 삽입할 컨텍스트 문자열.
    """
    lines = []
    if seed.get("product_name"): lines.append(f"제품명: {seed['product_name']}")
    if seed.get("brand"):        lines.append(f"브랜드: {seed['brand']}")
    if seed.get("category"):     lines.append(f"카테고리: {seed['category']}")
    if seed.get("price"):        lines.append(f"가격: {seed['price']}")
    if seed.get("description"):  lines.append(f"제품 설명 (발췌): {seed['description'][:200]}")
    return "\n".join(lines) if lines else "(메타 정보 없음)"


# 시연
if PRODUCT_SEEDS:
    print(build_product_context(PRODUCT_SEEDS[0]))

제품명: 클래식 라운드 안경
브랜드: muji
카테고리: 액세서리
제품 설명 (발췌): 클래식한 라운드 프레임의 안경으로 깔끔한 블랙 컬러가 특징입니다. 어떤 스타일에도 잘 어울리는 베이직한 디자인의 아이웨어입니다.


## S5 — 스타일 × 이미지 Vision 카피 프롬프트 + 1건 시연

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate


class AdCopy(BaseModel):
    """광고 카피 1건."""
    ad_copy:       str = Field(description="스타일에 맞는 한국어 광고 카피 1문장 (instruction 에서 요청한 스타일 기준)")
    confidence: int = Field(ge=1, le=5, description="스타일 일치도 자가 평가 1~5")
    reason:     str = Field(description="이 카피를 선택한 간단한 이유 (20자 이내)")


copy_parser = JsonOutputParser(pydantic_object=AdCopy)


SYSTEM_COPY = """당신은 한국 패션 쇼핑몰 전문 AI 카피라이터입니다.
제품 사진과 메타 정보를 바탕으로 지정된 브랜드 스타일에 맞는 한국어 광고 카피를 작성합니다.

작성 원칙:
- 사진에서 보이는 소재감·컬러·실루엣·착용감을 직접 반영
- 지정된 스타일 톤에서 절대 벗어나지 말 것
- 타 스타일 혼용 금지 (Style A 요청 시 B 톤 절대 사용 불가)

[스타일 톤 가이드]
{tone_hint}

[참고 예시 카피]
{examples}

{format_instructions}
"""

prompt_copy = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_COPY),
    ("user", [
        {"type": "text", "text":
            "[제품 정보]\n{product_context}\n\n"
            "[지시] 위 사진을 보고 '{style_name}' 스타일의 광고 카피를 작성하세요."},
        {"type": "image_url", "image_url": {"url": "{image_url}"}},
    ]),
]).partial(format_instructions=copy_parser.get_format_instructions())

chain_copy = prompt_copy | llm | copy_parser

print("[OK] 카피 생성 프롬프트 정의 완료")

[OK] 카피 생성 프롬프트 정의 완료


In [ ]:
def make_copy_args(seed: dict, style: StyleSpec, image_url: str) -> dict:
    """프롬프트 슬롯 채우기."""
    return {
        "tone_hint":       style.tone_hint,
        "examples":        "\n".join(f"- {ex}" for ex in style.examples),
        "product_context": build_product_context(seed),
        "style_name":      f"Style {style.style_id} ({style.name})",
        "image_url":       image_url,
    }


# 1건 시연 (첫 제품 + Style A)
if PRODUCT_SEEDS:
    seed0 = PRODUCT_SEEDS[0]
    style_a = STYLES["A"]
    demo_result = chain_copy.invoke(make_copy_args(seed0, style_a, data_urls[0]))
    print(f"[Style A — MUJI형]")
    print(f"  카피: {demo_result['ad_copy']}")
    print(f"  자가평가: {demo_result['confidence']}/5 | 이유: {demo_result['reason']}")
    print()

    style_b = STYLES["B"]
    demo_result_b = chain_copy.invoke(make_copy_args(seed0, style_b, data_urls[0]))
    print(f"[Style B — UNIQLO형 (LifeWear)]")
    print(f"  카피: {demo_result_b['ad_copy']}")
    print(f"  자가평가: {demo_result_b['confidence']}/5 | 이유: {demo_result_b['reason']}")

[Style A — MUJI형]
  카피: 시간이 지나도 변하지 않는 라운드 프레임의 기본입니다.
  자가평가: 5/5 | 이유: 기본과 시간성 강조

[Style B — UNIQLO형 (LifeWear)]
  카피: 매일 쓰고 싶은 안경이 있다면, 바로 이겁니다.
  자가평가: 5/5 | 이유: 일상 편안함을 감성적 표현


## S6 — 비동기 배치 생성 + 토큰/비용 측정

**N 제품 × 2 스타일 = 2N 건** 자동 생성.  
목표 450쌍: Style A 200 + Style B 200 + 혼합 50.

In [ ]:
semaphore = asyncio.Semaphore(3)   # 동시 API 호출 수 제한


async def gen_one_copy(seed: dict, style: StyleSpec, image_url: str, idx: int) -> dict | None:
    """1건 카피 비동기 생성."""
    async with semaphore:
        try:
            result = await chain_copy.ainvoke(make_copy_args(seed, style, image_url))
            return {
                "_idx":         idx,
                "product_id":   seed["product_id"],
                "product_name": seed["product_name"],
                "brand":        seed.get("brand", ""),
                "category":     seed.get("category", ""),
                "style_id":     style.style_id,
                "style_name":   style.name,
                # SFT 포맷 핵심 필드
                "instruction":  style.instruction_template,
                "output":       result["ad_copy"],
                "confidence":   result.get("confidence", 3),
                "reason":       result.get("reason", ""),
                "image_path":   seed["image_path"],
            }
        except Exception as e:
            print(f"  [{idx}] ERR: {e}")
            return None


print("[OK] 비동기 생성 함수 정의 완료")

[OK] 비동기 생성 함수 정의 완료


In [ ]:
# ── 전체 배치 생성 ───────────────────────────────────────────
# N_API_CALL: API 비용 절약 시 일부만. 전체는 len(PRODUCT_SEEDS)
# N_API_CALL = min(5, len(PRODUCT_SEEDS))
N_API_CALL = len(PRODUCT_SEEDS)   # 전체 74개

print(f"[INFO] API 호출 대상: {N_API_CALL}개 제품 × 2 스타일 = {N_API_CALL * 2}건")

t0 = time.time()
with make_cb_ctx() as cb:
    tasks = []
    for i in range(N_API_CALL):
        seed = PRODUCT_SEEDS[i]
        du   = data_urls[i]
        for sid, style in STYLES.items():
            tasks.append(gen_one_copy(seed, style, du, len(tasks)))
    raw = asyncio.run(asyncio.gather(*tasks))
elapsed = time.time() - t0

gen_results = [r for r in raw if r is not None]
print(f"\n[OK] 생성 {len(gen_results)}/{len(tasks)} 건  ({elapsed:.1f}s)")
print(f"  토큰 in:  {cb.prompt_tokens:,}")
print(f"  토큰 out: {cb.completion_tokens:,}")
print(f"  비용:     ${cb.total_cost:.4f}")

# 결과 출력
for r in gen_results[:6]:
    print(f"  [{r['style_id']}] {r['product_name'][:20]:<20} | {r['output']}")

[INFO] API 호출 대상: 74개 제품 × 2 스타일 = 148건
  [9] ERR: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 03f41e41-b841-480b-88a3-7ecb7600adf6, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011Cb1hLH83bSFqTDw5ebMhe'}
  [12] ERR: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 03f41e41-b841-480b-88a3-7ecb7600adf6, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-l

KeyboardInterrupt: 

In [ ]:
# ── Rate-limit 안전 배치 생성 ─────────────────────────────────
import math

CONCURRENCY   = 2        # 동시 API 호출 수 (3 → 2로 축소)
BATCH_SIZE    = 10       # 한 번에 처리할 최대 건수
BATCH_DELAY   = 15.0     # 배치 사이 대기 시간(초) — 토큰 리셋 여유

semaphore = asyncio.Semaphore(CONCURRENCY)

N_API_CALL = len(PRODUCT_SEEDS)   # 전체 74개
print(f"[INFO] API 호출 대상: {N_API_CALL}개 제품 × 2 스타일 = {N_API_CALL * 2}건")
print(f"[INFO] 배치 크기: {BATCH_SIZE}건, 배치 간 대기: {BATCH_DELAY}s")

# 전체 task 목록 구성
all_tasks_args = []
for i in range(N_API_CALL):
    seed = PRODUCT_SEEDS[i]
    du   = data_urls[i]
    for sid, style in STYLES.items():
        all_tasks_args.append((seed, style, du, len(all_tasks_args)))

# 배치 분할 실행
async def run_batched(tasks_args, batch_size, delay):
    all_results = []
    n_batches = math.ceil(len(tasks_args) / batch_size)
    for b_idx in range(n_batches):
        chunk = tasks_args[b_idx * batch_size : (b_idx + 1) * batch_size]
        print(f"  배치 {b_idx+1}/{n_batches} 시작 ({len(chunk)}건)...")
        batch_tasks = [gen_one_copy(s, st, du, idx) for s, st, du, idx in chunk]
        results = await asyncio.gather(*batch_tasks)
        all_results.extend(results)
        if b_idx < n_batches - 1:          # 마지막 배치는 대기 불필요
            print(f"  → {delay}초 대기 중...")
            await asyncio.sleep(delay)
    return all_results

t0 = time.time()
with make_cb_ctx() as cb:
    raw = asyncio.run(run_batched(all_tasks_args, BATCH_SIZE, BATCH_DELAY))
elapsed = time.time() - t0

gen_results = [r for r in raw if r is not None]
print(f"\n[OK] 생성 {len(gen_results)}/{len(all_tasks_args)} 건  ({elapsed:.1f}s)")
print(f"  토큰 in:  {cb.prompt_tokens:,}")
print(f"  토큰 out: {cb.completion_tokens:,}")
print(f"  비용:     ${cb.total_cost:.4f}")

for r in gen_results[:6]:
    print(f"  [{r['style_id']}] {r['product_name'][:20]:<20} | {r['output']}")

[INFO] API 호출 대상: 74개 제품 × 2 스타일 = 148건
[INFO] 배치 크기: 10건, 배치 간 대기: 15.0s
  배치 1/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 2/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 3/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 4/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 5/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 6/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 7/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 8/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 9/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 10/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 11/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 12/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 13/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 14/15 시작 (10건)...
  → 15.0초 대기 중...
  배치 15/15 시작 (8건)...

[OK] 생성 148/148 건  (511.4s)
  토큰 in:  285,552
  토큰 out: 10,337
  비용:     $1.0117
  [A] 클래식 라운드 안경           | 시간이 지나도 변하지 않는 라운드 프레임의 기본입니다.
  [B] 클래식 라운드 안경           | 매일 쓰고 싶은 안경이 있다면, 바로 이겁니다.
  [A] 엘라스틱 브레이드 벨트         | 신축성 있는 직조 소재로 허리에 부담 없이 편안합니다.
  [B] 엘라스틱 브레이드 벨트         | 매일 착용해도 편안한 신축성, 어떤 스타일에도 자연스럽게 어울립니다.
  [A] 대용량 폴딩 

## S7 — Claude-Judge 채점 (3축 rubric)

| 축 | 의미 |
|---|---|
| **style_fit** | 요청 스타일(MUJI/UNIQLO) 톤과 일치하는가 |
| **copy_quality** | 카피로서 완성도 (자연스러움, 문법, 길이) |
| **brand_match** | 제품 특성(소재·색상·카테고리)과 카피가 일치하는가 |

각 축 1~5점 → **통과 기준: 3축 평균 ≥ 4.0**

### MUJI vs UNIQLO 채점 포인트

| 체크 | MUJI형 통과 | UNIQLO형 통과 |
|------|------------|--------------|
| 톤   | 소재·기능 객관 서술 | 라이프스타일 감성 제안 |
| 어투 | 문어체 (~입니다) | 친근 구어체 (~이겁니다) |
| 감탄 | 없음 | 없음 (과장 금지) |
| 길이 | 30자 이내 | 35자 이내 |

In [ ]:
class JudgeScore(BaseModel):
    style_fit:     int = Field(ge=1, le=5, description="요청 스타일 톤 일치도")
    copy_quality:  int = Field(ge=1, le=5, description="카피 완성도 (간결성/자연스러움/설득력)")
    product_match: int = Field(ge=1, le=5, description="제품 특성 반영도")
    reason: str


judge_parser = JsonOutputParser(pydantic_object=JudgeScore)
judge_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한국 패션 광고 카피 품질 평가관입니다.\n"
     "아래 카피를 3축 rubric (각 1~5점) 으로 채점하세요.\n\n"
     "채점 기준:\n"
     "- style_fit: 요청 스타일 톤 준수 여부 (A=간결/기능, B=자신감/스트릿)\n"
     "- copy_quality: 카피 자체의 완성도 (간결·자연스러움·설득력)\n"
     "- product_match: 제품 특성이 카피에 녹아있는가\n\n"
     "{format_instructions}"
    ),
    ("user",
     "제품: {product_name} ({category})\n"
     "요청 스타일: Style {style_id} ({style_name})\n"
     "스타일 톤 가이드: {tone_hint}\n\n"
     "생성된 카피: {copy}"
    ),
]).partial(format_instructions=judge_parser.get_format_instructions())

judge_chain = judge_prompt | llm | judge_parser

PASS_THRESHOLD = 4
scored_results = []

for r in gen_results:
    style = STYLES[r["style_id"]]
    score = judge_chain.invoke({
        "product_name": r["product_name"],
        "category":     r["category"],
        "style_id":     r["style_id"],
        "style_name":   r["style_name"],
        "tone_hint":    style.tone_hint[:120],
        "copy":         r["output"],
    })
    combined = {**r, **score}
    scored_results.append(combined)
    status = "✅" if min(score["style_fit"], score["copy_quality"], score["product_match"]) >= PASS_THRESHOLD else "⚠️"
    print(f"{status} [{r['style_id']}] {r['product_name'][:18]:<18} | "
          f"style={score['style_fit']} quality={score['copy_quality']} match={score['product_match']} "
          f"| {score['reason'][:50]}")

n_pass = sum(1 for s in scored_results if min(s["style_fit"], s["copy_quality"], s["product_match"]) >= PASS_THRESHOLD)
print(f"\n통과율: {n_pass}/{len(scored_results)} 건 (임계: 모든 축 ≥ {PASS_THRESHOLD})")

✅ [A] 클래식 라운드 안경         | style=5 quality=5 match=4 | MUJI 스타일을 완벽하게 구현했습니다. 간결한 한 문장, 마침표 사용, 50자 이내(28
⚠️ [B] 클래식 라운드 안경         | style=4 quality=3 match=3 | UNIQLO 스타일의 '매일 입고 싶은' 표현을 '매일 쓰고 싶은'으로 잘 적용했고 친근한
✅ [A] 엘라스틱 브레이드 벨트       | style=5 quality=5 match=5 | MUJI 스타일 톤을 완벽하게 구현했습니다. '신축성 있는 직조 소재'로 제품의 핵심 특성
✅ [B] 엘라스틱 브레이드 벨트       | style=4 quality=4 match=5 | 유니클로 스타일의 핵심 요소인 '매일', '편안한', '자연스럽게 어울리는' 표현을 적절히
✅ [A] 대용량 폴딩 더플백         | style=5 quality=4 match=5 | MUJI 스타일을 완벽히 구현했습니다. 간결한 문장, 기능 중심 설명, 수식어 최소화, 마
⚠️ [B] 대용량 폴딩 더플백         | style=4 quality=4 match=3 | UNIQLO 스타일의 부드럽고 친근한 톤을 잘 구현했으며, '똑똑한'이라는 표현으로 감성적
✅ [A] 심플 에코백             | style=5 quality=5 match=4 | MUJI 스타일을 완벽히 구현한 카피입니다. '필요한 것만'이라는 미니멀한 철학과 '자연스
⚠️ [B] 심플 에코백             | style=4 quality=3 match=2 | 유니클로 스타일의 '매일 ~하고 싶은' 표현을 잘 활용하여 따뜻하고 친근한 톤을 구현했으나
✅ [A] 미니 백팩 키링 파우치       | style=5 quality=4 match=4 | MUJI 스타일을 완벽히 구현했습니다. 간결한 문장, 기능성 중심, 수식어 최소화, 마침표
⚠️ [B] 미니 백팩 키링 파우치       | style=4 quality

## S8 — DPO chosen/rejected 자동 생성 (Rejection Sampling)

**전략**: Negative Generation 방식  
- **chosen**: Judge 통과 카피 (style_fit ≥ 4)  
- **rejected**: 의도적으로 스타일 혼용한 카피 생성

### 혼용 패턴 예시

| rejected 유형 | 예시 |
|---|---|
| MUJI 요청에 UNIQLO 톤 | "매일 입고 싶은 편안함을 담았습니다" (감성 과잉) |
| UNIQLO 요청에 MUJI 톤 | "면 혼방 소재. 일상 활용에 적합합니다" (너무 건조) |
| 스트릿/직설 혼입 | "이거 하나면 진짜 완벽함" (스타일 이탈) |

→ rejected 생성 시 위 3가지 패턴을 랜덤 선택해 오염시킵니다.

In [ ]:
import random

# ✅ 수정 코드
NEGATIVE_SYSTEM_BASE = (
    "광고 카피를 작성하되, 다음 지시를 따르세요:\n"
    "{neg_pattern}\n\n"
    "{format_instructions}"
)

def make_negative_chain(style: StyleSpec):
    neg_pattern = random.choice(style.negative_patterns)

    neg_prompt = ChatPromptTemplate.from_messages([
        ("system", NEGATIVE_SYSTEM_BASE),
        ("user", [
            {"type": "text", "text":
                "[제품 정보]\n{product_context}\n\n"
                "[지시] '{style_name}' 스타일로 카피를 작성하세요. (하지만 위 부정 지시를 따르세요)"},
            {"type": "image_url", "image_url": {"url": "{image_url}"}},
        ]),
    ]).partial(                                          # ← .partial()로 체이닝
        neg_pattern=neg_pattern,
        format_instructions=copy_parser.get_format_instructions(),
    )

    return neg_prompt | llm_hot | copy_parser


async def gen_dpo_record(seed: dict, style: StyleSpec, chosen_copy: str, image_url: str) -> dict | None:
    """chosen (정상) + rejected (부정 패턴) 페어 생성."""
    args = make_copy_args(seed, style, image_url)
    neg_chain = make_negative_chain(style)
    async with semaphore:
        try:
            rejected_result = await neg_chain.ainvoke(args)
        except Exception as e:
            print(f"  [DPO ERR] {seed['product_id']} Style {style.style_id}: {e}")
            return None

    return {
        "product_id":  seed["product_id"],
        "style_id":    style.style_id,
        "style_name":  style.name,
        "image_path":  seed["image_path"],
        "image_rel":   seed.get("image_rel", f'images/{seed["brand"]}/{Path(seed["image_path"]).name}' if seed.get("brand") else seed["image_path"]),
        "prompt":      style.instruction_template,
        "chosen":      chosen_copy,
        "rejected":    rejected_result["ad_copy"],
        "system_role": f"Style {style.style_id} ({style.name}) 광고 카피라이터",
    }


print("[OK] DPO 페어 생성 함수 정의 완료")

[OK] DPO 페어 생성 함수 정의 완료


In [ ]:
# DPO 페어 생성 (통과한 SFT 결과 기반)
# N_DPO = min(4, len(scored_results))
N_DPO = len(scored_results)
dpo_candidates = [s for s in scored_results
                  if min(s["style_fit"], s["copy_quality"], s["product_match"]) >= PASS_THRESHOLD]
dpo_candidates = dpo_candidates[:N_DPO]
print(f"[INFO] DPO 페어 생성 대상: {len(dpo_candidates)}건")

t0 = time.time()
with make_cb_ctx() as cb_dpo:
    dpo_tasks = [
        gen_dpo_record(
            next(s for s in PRODUCT_SEEDS if s["product_id"] == cand["product_id"]),
            STYLES[cand["style_id"]],
            cand["output"],
            data_urls[next(i for i, s in enumerate(PRODUCT_SEEDS) if s["product_id"] == cand["product_id"])],
        )
        for cand in dpo_candidates
    ]
    dpo_raw = asyncio.run(asyncio.gather(*dpo_tasks))
elapsed = time.time() - t0

dpo_results = [r for r in dpo_raw if r is not None]
print(f"[OK] DPO 페어 {len(dpo_results)}/{len(dpo_candidates)} 건 생성 ({elapsed:.1f}s)")
print(f"  비용: ${cb_dpo.total_cost:.4f}")

if dpo_results:
    r = dpo_results[0]
    print(f"\n=== DPO 페어 미리보기 (Style {r['style_id']}) ===")
    print(f"chosen:   {r['chosen']}")
    print(f"rejected: {r['rejected']}")

[INFO] DPO 페어 생성 대상: 102건
  [DPO ERR] muji_top_003 Style A: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 03f41e41-b841-480b-88a3-7ecb7600adf6, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011Cb1kkryGzdAi8UvKUueKC'}
  [DPO ERR] muji_top_008 Style B: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 03f41e41-b841-480b-88a3-7ecb7600adf6, model: claude-sonnet-4-20250514). For details, refer to: https:

## S9 — dedup + jsonl 저장 + SFT 포맷 호환 검증

**저장 파일:**
- `adcopy_sft.jsonl` — SFT 학습용 (image, instruction, output)
- `adcopy_dpo.jsonl` — DPO preference 쌍 (V2 학습용)

### SFT instruction 포맷

```json
{
  "image": "images/tops_001.jpg",
  "instruction": "이 제품 사진을 보고 MUJI 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
  "output": "가볍고 부드러운 소재로 하루 종일 편안하게 입을 수 있습니다."
}
```

```json
{
  "image": "images/tops_001.jpg",
  "instruction": "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
  "output": "매일 입고 싶은 옷이 있다면, 바로 이겁니다."
}
```

In [ ]:
print(unique_sft[0].keys())

dict_keys(['_idx', 'product_id', 'product_name', 'brand', 'category', 'style_id', 'style_name', 'instruction', 'output', 'confidence', 'reason', 'image_path', 'style_fit', 'copy_quality', 'product_match'])


In [ ]:
from datetime import datetime

# ── n-gram Jaccard dedup ──────────────────────────────────────
def char_ngrams(s: str, n: int = 3) -> set:
    return {s[i:i+n] for i in range(len(s) - n + 1)} if len(s) >= n else {s}

def jaccard(a: str, b: str, n: int = 3) -> float:
    ga, gb = char_ngrams(a, n), char_ngrams(b, n)
    if not ga and not gb: return 0.0
    return len(ga & gb) / len(ga | gb)


# ── SFT 데이터 필터 + dedup ───────────────────────────────────
unique_sft = []
for s in scored_results:
    # 1. Judge 통과 필터
    if min(s["style_fit"], s["copy_quality"], s["product_match"]) < PASS_THRESHOLD:
        print(f"[skip] {s['product_id']} Style {s['style_id']}: Judge 미통과")
        continue
    # 2. dedup (output 기준 Jaccard)
    if any(jaccard(s["output"], u["output"]) >= 0.8 for u in unique_sft):
        print(f"[dup]  {s['product_id']} Style {s['style_id']}: 유사 카피 중복")
        continue
    unique_sft.append(s)

print(f"\nSFT 최종: {len(unique_sft)}건 (필터 + dedup 후)")

# ── SFT jsonl 저장 ────────────────────────────────────────────
SFT_OUT = "./adcopy_sft.jsonl"
with open(SFT_OUT, "w", encoding="utf-8") as f:
    for s in unique_sft:
        record = {
            "image":       s["image_path"],
            "instruction": s["instruction"],
            "output":      s["output"],
            "style_id":    s["style_id"],
            "style_name":  s["style_name"],
            "product_id":  s["product_id"],
            "brand":       s.get("brand", ""),
            "category":    s.get("category", ""),
            "meta": {
                "judge_scores": {
                    "style_fit":     s["style_fit"],
                    "copy_quality":  s["copy_quality"],
                    "product_match": s["product_match"],
                },
                "model": MODEL_NAME,
                "ts":    datetime.now().isoformat(),
            },
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"[OK] SFT 저장: {SFT_OUT}  ({len(unique_sft)} records)")

# ── DPO jsonl 저장 ────────────────────────────────────────────
DPO_OUT = "./adcopy_dpo.jsonl"
with open(DPO_OUT, "w", encoding="utf-8") as f:
    for r in dpo_results:
        record = {
            "image":    r["image_path"],
            "style_id": r["style_id"],
            "prompt":   r["prompt"],
            "system":   r["system_role"],
            "chosen":   r["chosen"],
            "rejected": r["rejected"],
            "meta":     {"model": MODEL_NAME, "ts": datetime.now().isoformat()},
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"[OK] DPO 저장: {DPO_OUT}  ({len(dpo_results)} records)")

[skip] muji_acc_001 Style B: Judge 미통과
[skip] muji_bag_001 Style B: Judge 미통과
[skip] muji_bag_002 Style B: Judge 미통과
[skip] muji_bag_003 Style B: Judge 미통과
[skip] muji_bag_004 Style B: Judge 미통과
[skip] muji_bag_005 Style B: Judge 미통과
[skip] muji_bottom_003 Style B: Judge 미통과
[skip] muji_bottom_004 Style B: Judge 미통과
[skip] muji_bottom_006 Style B: Judge 미통과
[skip] muji_outer_001 Style B: Judge 미통과
[skip] muji_outer_002 Style B: Judge 미통과
[skip] muji_outer_003 Style B: Judge 미통과
[skip] muji_outer_004 Style B: Judge 미통과
[skip] muji_outer_005 Style B: Judge 미통과
[skip] muji_outer_006 Style B: Judge 미통과
[skip] muji_outer_007 Style B: Judge 미통과
[skip] muji_outer_008 Style B: Judge 미통과
[skip] muji_outer_009 Style B: Judge 미통과
[skip] muji_outer_010 Style B: Judge 미통과
[skip] muji_top_003 Style B: Judge 미통과
[skip] muji_top_005 Style B: Judge 미통과
[skip] muji_top_006 Style B: Judge 미통과
[skip] muji_top_007 Style B: Judge 미통과
[skip] muji_top_009 Style B: Judge 미통과
[skip] uniqlo_acc_001 Style A: Judg

In [ ]:
# ── SFT 포맷 호환 검증 ────────────────────────────────────────
print("[SFT 포맷 검증]")
with open(SFT_OUT, encoding="utf-8") as f:
    records = [json.loads(l) for l in f]

required_keys = {"image", "instruction", "output"}
errors = []
for i, rec in enumerate(records):
    missing = required_keys - set(rec.keys())
    if missing:
        errors.append(f"  #{i}: 필수 키 누락 {missing}")
    if not rec.get("output"):
        errors.append(f"  #{i}: output 비어있음")

if errors:
    print("❌ 오류 발견:")
    for e in errors: print(e)
else:
    print(f"✅ 전체 {len(records)}건 포맷 이상 없음")
    style_dist = {}
    for rec in records:
        sid = rec.get("style_id", "?")
        style_dist[sid] = style_dist.get(sid, 0) + 1
    print(f"   스타일 분포: {style_dist}")

[SFT 포맷 검증]
✅ 전체 95건 포맷 이상 없음
   스타일 분포: {'A': 68, 'B': 27}


## S10 — 비용 외삽 (450쌍 전체 생성 예상)

목표: SFT 450쌍 + DPO preference 250쌍

In [ ]:
n_actual = len(gen_results)
if n_actual > 0:
    cost_per_sft = cb.total_cost / n_actual
    cost_per_dpo = (cb_dpo.total_cost / len(dpo_results)) if dpo_results else cost_per_sft * 2

    print(f"실측 SFT 1건당:  ${cost_per_sft:.4f}")
    print(f"실측 DPO 1건당:  ${cost_per_dpo:.4f}")
    print()
    print(f"  {'규모':>22} | {'SFT 비용':>10} | {'DPO 비용':>10} | {'합계':>10}")
    print(f"  {'-'*22} | {'-'*10} | {'-'*10} | {'-'*10}")
    for sft_n, dpo_n, label in [
        (50,  25,  "이미지 25개 × 2style"),
        (100, 50,  "이미지 50개 × 2style"),
        (200, 100, "이미지 100개 확장"),
        (350, 175, "이미지 175개 확장"),
        (450, 250, "목표 전체 450쌍"),
    ]:
        sc = cost_per_sft * sft_n
        dc = cost_per_dpo * dpo_n
        print(f"  {label:<22} | ${sc:>8.3f} | ${dc:>8.3f} | ${sc+dc:>8.3f}")
    print()
    print(f"[참고] 모델: {MODEL_NAME}  (in ${PRICE_IN}/1M, out ${PRICE_OUT}/1M)")

---

## 정리 — Day11 SFT 연결 가이드

| 이 노트북 산출물 | Day11 연결 |
|---|---|
| `adcopy_sft.jsonl` | Qwen2.5-VL SFT 학습 데이터 입력 |
| `adcopy_dpo.jsonl` | DPO preference 학습 데이터 (V2) |
| `images/*.jpg`     | 학습 시 이미지 경로 그대로 사용 |

### 450쌍 확보 전략 (유니클로 이미지 기준)
```
현재:  이미지 50개 × 2 스타일 = 100쌍 (필터 후 ~80쌍 예상)
목표:  이미지 225개+ 확보 시 450쌍 달성 가능

카테고리 균형 권장 (유니클로 제품 기준):
  상의(tops)    50장  ← 히트텍, 에어리즘, 후리스 등 주력
  하의(bottoms) 30장  ← 울트라스트레치, 슬랙스 등
  아우터(outer) 30장  ← 블록텍, 패딩, 카디건 등
  기타(shoes/acc/bag) 40장
  합계          150장 → SFT 300쌍 (V1 최소 목표)
```

### 유니클로 이미지 수집 방법
```
1. 유니클로 공식몰 (uniqlo.com/kr) → 제품 상세 페이지
2. 대표 이미지 우클릭 → 이미지 저장 (또는 개발자도구 → Network)
3. 파일명 규칙 준수: {category}_{번호}.jpg
   (tops_001.jpg, bottoms_001.jpg, outer_001.jpg ...)
4. 이미지 크기: 400px 이상 권장 (Claude Vision 정확도)
5. 단품 이미지 권장 (착용 모델 이미지도 가능)
```

### 실험 변수 (Day11에서 고정)
```
변수 1: QLoRA rank   → 8 vs 16
변수 2: 학습 데이터  → 200쌍 vs 350쌍
베이스 모델: Qwen2.5-VL
```

### SMART 목표 (변경 없음)
```
AS-IS: Qwen2.5-VL base Claude-judge 스타일 일치도 40/100
TO-BE: SFT 후 60/100 이상 (+20점)
측정:  hold-out 100개, Claude-judge 5점 척도
```

In [ ]:
!pwd
!ls


/content
adcopy_dpo.jsonl  adcopy_sft.jsonl  drive  images  muji  sample_data  uniqlo


In [ ]:
!ls


adcopy_dpo.jsonl  adcopy_sft.jsonl  drive  images  muji  sample_data  uniqlo


In [ ]:
save_dir = '/content/drive/MyDrive/adcopy_data'
os.makedirs(save_dir, exist_ok=True)

# 파일 복사
files_to_save = ['adcopy_sft.jsonl', 'adcopy_dpo.jsonl']

for filename in files_to_save:
    src = f'/content/{filename}'
    dst = os.path.join(save_dir, filename)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ 저장 완료: {dst}')
    else:
        print(f'⚠️ 파일 없음: {src}')

print('\n📁 저장 위치:', save_dir)

✅ 저장 완료: /content/drive/MyDrive/adcopy_data/adcopy_sft.jsonl
✅ 저장 완료: /content/drive/MyDrive/adcopy_data/adcopy_dpo.jsonl

📁 저장 위치: /content/drive/MyDrive/adcopy_data
